# Tests for `plotter.py`

Run this notebook from the folder it lives in (next to `plotter.py`).
`plotter.py` locates the dataset on its own, so the data path does not depend
on the working directory.

Cells ending in `PASSED` check a fact automatically (they raise an
`AssertionError` if something is wrong). The plots at the end are visual checks.

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.getcwd())   # import plotter.py from this notebook's own folder
import numpy as np
import pandas as pd
import plotter

## 1. Metadata and helpers
These need no data file and run instantly.

In [ ]:
cols = plotter.imu_cols('hand')
assert len(cols) == 17
assert cols[0] == 'hand_temp'
assert 'hand_acc16_x' in cols and 'hand_gyro_z' in cols
print('imu_cols: PASSED')

In [ ]:
assert len(plotter.COLUMNS) == 54
assert plotter.COLUMNS[:3] == ['timestamp', 'activity_id', 'heart_rate']
print('COLUMNS: PASSED (54 columns)')

In [ ]:
assert plotter.NAME_TO_ID['walking'] == 4
assert plotter.ACTIVITIES[plotter.NAME_TO_ID['running']] == 'running'
print('NAME_TO_ID round-trip: PASSED')

## 2. Loading a subject

In [ ]:
df = plotter.load_subject('101')
assert df.shape[1] == 54
assert len(df) > 0
known = set(plotter.ACTIVITIES) | {0}   # 0 = transient
assert set(df['activity_id'].unique()).issubset(known)
print('load_subject: PASSED', df.shape)

## 3. `get_segment` behaviour
Correct window size, the `start` offset, and the three ways it should decline.
These three ways are `unknown activity`, `activity not performed`, `window past the end`.

In [ ]:
seg = plotter.get_segment(df, 'lying', start=0, seconds=10)
assert seg is not None
assert len(seg) == 10 * plotter.SAMPLE_RATE   # 10 s at 100 Hz
print('window length: PASSED', len(seg), 'rows')

In [ ]:
seg0  = plotter.get_segment(df, 'lying', start=0,  seconds=5)
seg10 = plotter.get_segment(df, 'lying', start=10, seconds=5)
assert seg10['timestamp'].iloc[0] > seg0['timestamp'].iloc[0]
print('start offset: PASSED')

In [ ]:
assert plotter.get_segment(df, 'skydiving', 0, 10) is None
print('unknown activity -> None: PASSED')

In [ ]:
present = set(df['activity_id'].unique())
missing = [name for aid, name in plotter.ACTIVITIES.items() if aid not in present]
if missing:
    assert plotter.get_segment(df, missing[0], 0, 10) is None
    print('not-performed activity -> None: PASSED  (tested', repr(missing[0]) + ')')
else:
    print('subject did every activity; nothing to test here')

In [ ]:
assert plotter.get_segment(df, 'lying', start=999999, seconds=10) is None
print('window past end -> None: PASSED')

## 4. `compute_spectrum`
The FFT is its own function returning `(freqs, spectrum)`, so it can be tested
directly. A pure 3 Hz sine sampled at 100 Hz should peak at 3 Hz.

In [ ]:
fs = plotter.SAMPLE_RATE
t = np.arange(0, 10, 1 / fs)          # 10 seconds at 100 Hz
signal = np.sin(2 * np.pi * 3 * t)    # pure 3 Hz sine

freqs, spectrum = plotter.compute_spectrum(signal)
peak = freqs[np.argmax(spectrum)]
assert abs(peak - 3) < 0.2
print('compute_spectrum finds the 3 Hz peak: PASSED  (peak =', round(peak, 2), 'Hz)')

In [ ]:
# Punch a gap of NaNs into the same signal; compute_spectrum should clean it
# internally so the output has no NaN and the 3 Hz peak survives.
gappy = signal.copy()
gappy[100:110] = np.nan
freqs2, spec2 = plotter.compute_spectrum(gappy)
assert not np.isnan(spec2).any()
peak2 = freqs2[np.argmax(spec2)]
assert abs(peak2 - 3) < 0.2
print('compute_spectrum handles NaN gaps: PASSED  (peak =', round(peak2, 2), 'Hz)')

## 5. Visual checks
Eyeball these. Lying flat, running busy; the running spectrum should carry far
more energy in the 1-8 Hz band than lying.

In [ ]:
plotter.plot_activity('101', 'lying')

In [ ]:
plotter.plot_activity('101', 'running')

In [ ]:
plotter.plot_activity('101', 'walking', start=30, seconds=10)   # offset window

In [ ]:
plotter.plot_activity('101', 'running', magnitude=True)         # movement magnitude

### Sample points and the interpolator
`markers=True` draws a dot on each sample. `interpolate=True` reads the data through
the same interpolator the feature extraction and ML pipeline use
(`HeartbeatDataProcessor._interpolate_df`), which linearly fills NaN gaps shorter than
`max_interp_length` seconds and leaves longer gaps as breaks. Combined, a filled gap
shows as evenly spaced dots on a straight line between the two real samples bracketing
it, so an interpolated run is distinguishable from genuine motion. Both flags are on
`plot_activity`, `plot_activity_multi`, and `plot_subjects_multi`.

In [ ]:
# A short window with markers on, so individual samples are visible.
plotter.plot_activity('101', 'walking', start=30, seconds=2, markers=True)

# The same window read through the pipeline interpolator. Markers on a filled gap fall
# on a straight segment between the two real samples bracketing it.
plotter.plot_activity('101', 'walking', start=30, seconds=2, markers=True, interpolate=True)

In [ ]:
# Both flags work on the multi-panel plots too: two sensors, markers on, interpolated.
plotter.plot_activity_multi('101', 'walking', sensors=('hand_acc16', 'ankle_acc16'),
                            start=30, seconds=2, markers=True, interpolate=True)

# One sensor across two subjects, markers on, interpolated.
plotter.plot_subjects_multi(('101', '102'), 'walking', starts=(30, 30), seconds=2,
                            markers=True, interpolate=True)

### Single vs. multiple sensors
The four checks above show one sensor at a time; these show several sensors
side by side via plot_activity_multi.

In [ ]:
plotter.plot_activity_multi('101', 'running', sensors=('hand_acc16', 'ankle_acc16'))  # two sensors side by side

In [ ]:
plotter.plot_activity_multi('101', 'walking', sensors=('hand_acc16', 'chest_acc16', 'ankle_acc16'), magnitude=True)  # three sensors, magnitude

### Multiple participants
plot_subjects_multi puts one participant per panel, each with its own
window start.

In [ ]:
plotter.plot_subjects_multi(('101', '102'), 'walking', starts=(0, 30))  # per-subject start

In [ ]:
plotter.plot_subjects_multi(('101', '102', '103'), 'running', sensor='ankle_acc16', magnitude=True)

In [ ]:
plotter.plot_spectrum('101', 'running')

In [ ]:
plotter.plot_spectrum('101', 'lying')

## 6. Plotting functions — parameters & defaults

**Activity-window plots** (`plot_activity`, `plot_activity_multi`, `plot_subjects_multi`, `plot_spectrum`, `plot_spectrum_multi`) share these, with defaults shown:

- `subject="101"` — participant id (reads `subject101.dat`).
- `activity="walking"` — activity name to isolate.
- `sensor="hand_acc16"` — sensor prefix; `_x/_y/_z` are appended automatically.
- `start=10` — **seconds into the interval** to begin (default skips the noisy onset of the activity).
- `seconds=10` — window length in seconds.
- `magnitude=False` — `False` plots x/y/z; `True` plots √(x²+y²+z²). *(not on the spectrum plots)*
- `activity_interval=0` — which occurrence of the activity to use, 0 = first. *(only `plot_activity` and `plot_spectrum`)*

**Time-domain extras** (`plot_activity`, `plot_activity_multi`, `plot_subjects_multi`):
- `markers=False` — `True` draws a dot on each sample.
- `interpolate=False` — `True` reads the data through the ML pipeline's interpolator (`HeartbeatDataProcessor._interpolate_df`), which fills NaN gaps shorter than `max_interp_length` seconds; it applies only to the automatic read, not to a supplied `df`. With `markers=True` an interpolated gap reads as dots on a straight segment between the two real samples bracketing it.

**Per-function extras:**
- `plot_activity_multi(sensors=("hand_acc16","chest_acc16"))` — tuple of sensors, one panel each.
- `plot_subjects_multi(subjects=("101","102"), starts=10)` and `plot_spectrum_multi(...)` — tuple of subjects, one panel each; `starts` is one value for all **or** one per subject.
- `plot_spectrum(max_freq=15)` / `plot_spectrum_multi(max_freq=15)` — upper x-axis limit in Hz.
- `plot_timeseries(subject="101", columns=("heart_rate",), start=None, end=None)` — plots any column(s) vs the raw timestamp, **no activity filter**; `start`/`end` are timestamps in seconds (`None` = whole recording).

`load_subject(subject, interpolate=False, max_interp_length=0.1)` — reads one subject's recording; `interpolate=True` runs the pipeline interpolator over it.

`compute_spectrum(values, sample_rate=100)` — the FFT helper used by the spectrum plots; returns `(freqs, spectrum)`.

## 7. Examples by measurement type
One plot per kind of measurement in the dataset, so each channel type has a worked example.

In [ ]:
# Accelerometer x/y/z (±16g) — the three axes as separate lines
plotter.plot_activity('101', 'walking', sensor='ankle_acc16')

In [ ]:
# Magnitude — the same accelerometer collapsed to √(x²+y²+z²)
plotter.plot_activity('101', 'running', sensor='ankle_acc16', magnitude=True)

In [ ]:
# Heart rate — by timestamp, no activity filter (sparse channel, NaNs dropped)
plotter.plot_timeseries('101', 'heart_rate')

In [ ]:
# Temperature — a scalar channel, also via plot_timeseries
plotter.plot_timeseries('101', 'hand_temp')

In [ ]:
# Gyroscope x/y/z — rotation rate
plotter.plot_activity('101', 'running', sensor='hand_gyro')

In [ ]:
# Magnetometer x/y/z
plotter.plot_activity('101', 'walking', sensor='chest_mag')

In [ ]:
# Frequency content (FFT) of an accelerometer
plotter.plot_spectrum('101', 'running', sensor='ankle_acc16')

In [ ]:
# FFT across subjects — one panel each
plotter.plot_spectrum_multi(('101', '102'), 'running')